In [1]:
import cv2
import numpy as np

def contar_objetos_marrom_circulares(caminho_imagem, circularidade_min, area_min, mostrar_resultado):
    """
    Conta o número de objetos marrons e circulares em uma imagem.

    Parâmetros:
    - caminho_imagem (str): O caminho para o arquivo de imagem.
    - circularidade_min (float): O valor mínimo do índice de circularidade
                                 para considerar um objeto como circular (perfeito é 1.0).
    - area_min (int): A área mínima do contorno (em pixels) para filtrar objetos pequenos.

    Retorna:
    - int: O número de objetos que atendem aos critérios.
    """
    
    # 1. Carregar a imagem
    imagem_bgr = caminho_imagem
    if imagem_bgr is None:
        print(f"ERRO: Não foi possível carregar a imagem em {caminho_imagem}")
        return 0

    # 2. Converter para o espaço de cores HSV (melhor para segmentação de cor)
    imagem_hsv = cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2HSV)

    # 3. Definir o intervalo de cores para marrom em HSV
    # Marrom é um tom de laranja/vermelho escuro e dessaturado.
    # [H_min, S_min, V_min] e [H_max, S_max, V_max]
    # Faixa comum para marrom: H: 10-30, S: 50-200, V: 20-150
    lower_brown = np.array([10, 40, 70])
    upper_brown = np.array([30, 150, 255])

    # 4. Criar uma máscara de cor marrom
    mascara_cor = cv2.inRange(imagem_hsv, lower_brown, upper_brown)

    # 5. Aplicar operações morfológicas para limpar a máscara (remover ruído)
    kernel = np.ones((7, 7), np.uint8)
    mascara_limpa = cv2.morphologyEx(mascara_cor, cv2.MORPH_OPEN, kernel)
    mascara_limpa = cv2.morphologyEx(mascara_limpa, cv2.MORPH_CLOSE, kernel)

    # 6. Encontrar contornos na máscara
    # O contorno representa o limite (borda) dos objetos marrons
    contornos, _ = cv2.findContours(mascara_limpa, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    objetos_contados = 0
    imagem_resultado = imagem_bgr.copy() # Cópia para desenhar o resultado (opcional)

    # 7. Iterar sobre os contornos e verificar circularidade
    for contorno in contornos:
        area = cv2.contourArea(contorno)

        # 7a. Filtrar contornos muito pequenos (ruído)
        if area < area_min:
            continue

        # 7b. Calcular o perímetro para verificar a circularidade
        perimetro = cv2.arcLength(contorno, True)

        # Evitar divisão por zero
        if perimetro == 0:
            continue

        # Índice de Circularidade: C = 4 * pi * (Area / Perimetro^2)
        # Para um círculo perfeito, C = 1.0.
        circularidade = (4 * np.pi * area) / (perimetro ** 2)

        # 7c. Verificar se o objeto é circular o suficiente
        if circularidade >= circularidade_min:
            objetos_contados += 1
            
            # Opcional: Desenhar o contorno e o círculo envolvente no resultado
            cv2.drawContours(imagem_resultado, [contorno], -1, (0, 255, 0), 2)
            (x, y), raio = cv2.minEnclosingCircle(contorno)
            centro = (int(x), int(y))
            cv2.circle(imagem_resultado, centro, int(raio), (0, 0, 255), 3)
            cv2.putText(imagem_resultado, f"Circularidade: {circularidade:.2}", (centro[0] + 20, centro[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255),2)

    #Inicio mostrar resultado de figura
    if mostrar_resultado:
        cv2.imshow("Máscara Limpa (Marrom)", mascara_limpa)
        cv2.imshow("Objeto Identificado", imagem_resultado)
        
    #Fim do mostrar resultado de figura
    
    # Opcional: Salvar ou exibir a imagem com os objetos identificados
    # cv2.imwrite('resultado_contagem.png', imagem_resultado)
    #cv2.imshow('Objetos Identificados', imagem_resultado)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()
    
    return objetos_contados, imagem_resultado


In [2]:

#Captura do video
cap = cv2.VideoCapture(0)

#Criar uma janela para o video
cv2.namedWindow('Janela')

while True:
    
    #Capturar frame a frame
    ret, frame = cap.read()

    contagem, resultado = contar_objetos_marrom_circulares(frame, circularidade_min=0.65, area_min=100, mostrar_resultado=True)
    print(f"Total de objetos marrons circulares encontrados: {contagem}")

   
    #Apresentar o frame resultante
    #cv2.imshow('Janela', frame)
    
    #Comando de saída
    if cv2.waitKey(1) & 0xFF == ord('s'):
        break
    
    
#Quando finalizar, destruir os elementos
cap.release()
cv2.destroyAllWindows()

Total de objetos marrons circulares encontrados: 1
Total de objetos marrons circulares encontrados: 1
Total de objetos marrons circulares encontrados: 2
Total de objetos marrons circulares encontrados: 4
Total de objetos marrons circulares encontrados: 4
Total de objetos marrons circulares encontrados: 2
Total de objetos marrons circulares encontrados: 1
Total de objetos marrons circulares encontrados: 2
Total de objetos marrons circulares encontrados: 3
Total de objetos marrons circulares encontrados: 1
Total de objetos marrons circulares encontrados: 0
Total de objetos marrons circulares encontrados: 1
Total de objetos marrons circulares encontrados: 0
Total de objetos marrons circulares encontrados: 2
Total de objetos marrons circulares encontrados: 1
Total de objetos marrons circulares encontrados: 0
Total de objetos marrons circulares encontrados: 2
Total de objetos marrons circulares encontrados: 0
Total de objetos marrons circulares encontrados: 1
Total de objetos marrons circul